In [ ]:
import json
from pathlib import Path

ENTS_VAL = PROJECT_ROOT / "data" / "processed" / "entidades_gliner_validated"
ENTS_VAL.mkdir(parents=True, exist_ok=True)

for viejo in ENTS_VAL.glob("*.json"):
    viejo.unlink()

for ruta in sorted(ENTIDADES_DIR.glob("*_candidates.json")):
    data   = json.loads(ruta.read_text(encoding="utf-8"))
    doc_id = ruta.stem.replace("_candidates", "")

    # Agrupa por (texto normalizado, etiqueta) para no repetir decisiones
    vistos, decisions = set(), []
    for e in data["gliner_candidates"]:
        texto = e["text"].strip()
        label = e["label"].upper()
        clave = (texto.lower(), label)
        if not texto or clave in vistos:
            continue
        vistos.add(clave)
        decisions.append({
            "text":           texto,
            "canonical_form": texto,
            "final_label":    label,
            "confirmed":      True,
        })

    payload = {"doc_id": doc_id, "decisions": decisions}
    (ENTS_VAL / f"{doc_id}_validated.json").write_text(
        json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")

    print(f"  ✓ {doc_id:<30} {len(decisions):>3} decisiones")

In [ ]:
from src.anonymization.replacer import EntityReplacer

OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "entrevistas_anonimizadas_gliner"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

replacer = EntityReplacer(symbol="brackets")
results = replacer.anonymize_directory(
    validated_dir = ENTS_VAL,
    originals_dir = ORIGINAL_DIR,
    output_dir    = OUTPUT_DIR,
)

In [ ]:
from src.anonymization.replacer import EntityReplacer

OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "entrevistas_anonimizadas_gliner"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

replacer = EntityReplacer(symbol="brackets")
results = replacer.anonymize_directory(
    validated_dir = ENTS_VAL,
    originals_dir = ORIGINAL_DIR,
    output_dir    = OUTPUT_DIR,
)

In [ ]:
generados = sorted(OUTPUT_DIR.glob("*_anonimizado.txt"))
print(f"Generados: {len(generados)}\n")
if generados:
    print(generados[0].read_text(encoding="utf-8")[:1500])